<a href="https://colab.research.google.com/github/ohki-yu0225/toukei_I/blob/main/261023_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install japanize-matplotlib

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib

plt.rcParams["figure.figsize"] = (4, 3)
%config InlineBackend.figure_format = "retina"

file_id = "17rKrgPe8D2LadSw1YQkOVp3YvTeiT0ak"

gdown.download(
    id=file_id,
    output="room.csv",
    quiet=False
)

df = pd.read_csv("room.csv", encoding="shift-jis")
df

### 平均値・標準偏差・分散

平均値は広く用いられる位置の代表値である。観測値の合計を個数で割った値として定義される。標準偏差と分散は平均値からのデータの広がりを表す尺度である。


In [ ]:
mean = df["家賃"].mean()
std = df["家賃"].std()
var = df["家賃"].var()

print("平均値:", round(mean, 1), "円")
print("標準偏差:", round(std, 1), "円")
print("分散:", round(var, 1), "円^2")

In [ ]:
bins = range(60000, 150001, 10000)

plt.figure(figsize=(5, 3))
plt.hist(df["家賃"], bins=bins, color="#c8c8c8", edgecolor="black")
plt.axvline(mean, color="#c81e1e", label="平均値", linewidth=2)
plt.axvline(mean - std, color="#0050b3", linestyle="--", label="平均値±標準偏差", linewidth=2)
plt.axvline(mean + std, color="#0050b3", linestyle="--", linewidth=2)
plt.xlabel("家賃")
plt.ylabel("度数")
plt.legend()
plt.show()

### 中央値・最頻値

平均値は外れ値の影響を受けやすいという特徴がある。そのため，平均値以外の代表値である中央値や最頻値が用いられる場合がある。

In [ ]:
median = df["家賃"].median()

counts, edges = np.histogram(df["家賃"], bins=bins)
i = counts.argmax()
mode = (edges[i] + edges[i + 1]) / 2

print("中央値:", round(median, 1), "円")
print("最頻値:", round(mode, 1), "円")

### 範囲・四分位範囲

観測値を小さい順に並べ，4分の1ずつの場所にある値をそれぞれ第1四分位数，第2四分位数（中央値と等しい），第3四分位数という。四分位範囲は第3四分位数と第1四分位数の差であり，広がりの尺度の一種である。

In [ ]:
q1 = df["家賃"].quantile(0.25)
q2 = df["家賃"].quantile(0.50)
q3 = df["家賃"].quantile(0.75)
iqr = q3 - q1

print("第1四分位数:", round(q1, 1), "円")
print("中央値（第2四分位数）:", round(q2, 1), "円")
print("第3四分位数:", round(q3, 1), "円")
print("四分位範囲:", round(iqr, 1), "円")

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(df["家賃"], bins=bins, color="#c8c8c8", edgecolor="black")
plt.axvline(q1, color="#0050b3", linestyle="--", label="第1・第3四分位数", linewidth=2)
plt.axvline(q2, color="#c81e1e", label="中央値（第2四分位数）", linewidth=2)
plt.axvline(q3, color="#0050b3", linestyle="--", linewidth=2)
plt.xlabel("家賃")
plt.ylabel("度数")
plt.legend()
plt.show()

### 5数要約・箱ひげ図

最小値，第1四分位数，第2四分位数（中央値），第3四分位数，最大値を5数といい，これらをまとめた表を5数要約という。5数要約を視覚的に利用してデータを表現したものに箱ひげ図がある。

In [ ]:
df["家賃"].describe()

In [ ]:
plt.boxplot(df["家賃"])
plt.xticks([1], [""])
plt.ylabel("家賃")
plt.show()

In [ ]:
rent_A = df[df["近さ"] == "A"]["家賃"]
rent_B = df[df["近さ"] == "B"]["家賃"]

plt.boxplot([rent_A, rent_B])
plt.xticks([1, 2], ["A", "B"])
plt.xlabel("近さ")
plt.ylabel("家賃")
plt.show()

### 演習：右すその歪んだ分布での代表値

右すその歪んだ分布では，平均値・中央値・最頻値が乖離し，通常，最頻値<中央値<平均値という大小関係を持つ。下のコードの `skewness`（右すその長さを表すパラメータ）の値を実行し，右すそが長くなるほど3つの代表値の差がどう変わるかを確かめよう。

In [ ]:
# ここを変えてみよう：右裾の長さ（歪度）：0なら左右対称，大きいほど右裾が長い（0以上2以下）
skewness = 0

n = 10000

# 左右対称な分布：平均50，標準偏差15の正規分布
np.random.seed(0)
sym = np.random.normal(50, 15, n)

# 右裾に歪んだ分布：正規分布に指数分布を足す（平均50，標準偏差15）
tau = 15 * (skewness / 2) ** (1 / 3)
sigma = np.sqrt(15**2 - tau**2)
skew = np.random.normal(50 - tau, sigma, n) + np.random.exponential(tau, n)

bins2 = np.arange(-25, 206, 10)

# 最頻値（度数が最も大きい階級の代表値）
counts, edges = np.histogram(sym, bins=bins2)
i = counts.argmax()
sym_mode = (edges[i] + edges[i + 1]) / 2

counts, edges = np.histogram(skew, bins=bins2)
i = counts.argmax()
skew_mode = (edges[i] + edges[i + 1]) / 2

print("skewness =", skewness)
print("左右対称な分布  平均値:", round(np.mean(sym), 1), " 中央値:", round(np.median(sym), 1), " 最頻値:", round(sym_mode, 1))
print("右裾に歪んだ分布 平均値:", round(np.mean(skew), 1), " 中央値:", round(np.median(skew), 1), " 最頻値:", round(skew_mode, 1))

plt.figure(figsize=(8, 3))

plt.subplot(1, 2, 1)
plt.hist(sym, bins=bins2, color="#c8c8c8", edgecolor="black")
plt.axvline(np.mean(sym), color="#c81e1e", label="平均値", linewidth=1.5)
plt.axvline(np.median(sym), color="#0050b3", linestyle="--", label="中央値", linewidth=1.5)
plt.axvline(sym_mode, color="#00754e", linestyle=":", label="最頻値", linewidth=1.5)
plt.title("左右対称な分布")
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(skew, bins=bins2, color="#c8c8c8", edgecolor="black")
plt.axvline(np.mean(skew), color="#c81e1e", label="平均値", linewidth=1.5)
plt.axvline(np.median(skew), color="#0050b3", linestyle="--", label="中央値", linewidth=1.5)
plt.axvline(skew_mode, color="#00754e", linestyle=":", label="最頻値", linewidth=1.5)
plt.title("右裾に歪んだ分布（skewness=" + str(skewness) + "）")
plt.legend()

plt.tight_layout()
plt.show()